# 08 — Group GEMM

**Ladder:** Sequential `torch.matmul` → sequential tutorial-03 Triton matmul → (official) one-launch Group GEMM with a pointer table.

**Study tip.** Tutorial-03 **grouped L2 pid** ≠ **Group GEMM**. This notebook’s Triton path is the sequential baseline; the pointer-table one-launch design is the next mental step.

Official: [Group GEMM](https://triton-lang.org/main/getting-started/tutorials/08-group-gemm.html)

Primer: [`../03_matrix_multiplication/README.md`](../03_matrix_multiplication/README.md) · figures: [`docs/images/`](docs/images/)

![Whiteboard: $G$ host launches vs one group-GEMM pointer table](docs/images/benchmark-group-gemm.png)

*Theory board — sequential launches vs device arrays of pointers + $(M_g,N_g,K_g)$ in one kernel.*

---

## 0. Problem family

A **group** is a list of independent products

$$
C^{(g)} = A^{(g)} B^{(g)},
\qquad
A^{(g)} \in \mathbb{R}^{M_g \times K_g},\;
B^{(g)} \in \mathbb{R}^{K_g \times N_g},
\quad g = 1,\ldots,G.
$$

**Total FLOPs.**

$$
\sum_{g=1}^{G} 2 M_g N_g K_g.
$$

**Launch cost model (sketch).** Sequential launches pay $\approx G \cdot T_{\mathrm{launch}}$ plus compute. When problems are small, $G \cdot T_{\mathrm{launch}}$ dominates. A single-launch Group GEMM packs metadata so one grid covers all $g$.

**Naming note.** Tutorial 03's "grouped" ordering is an **L2 swizzle inside one GEMM**. Group GEMM is a **batch of many GEMMs** (often different shapes).

**Contrast with `torch.bmm`.** Batched matmul needs identical $(M,N,K)$ per item. MoE / multi-LoRA need heterogeneous $M_g$ → Group GEMM (or wasteful pad+mask).

**Teaching shapes (below).** $(128\times 64)(64\times 256)$ and $(64\times 128)(128\times 32)$.

### Theory board

![Whiteboard: $G$ host launches vs one group-GEMM pointer table](docs/images/whiteboard-theory.jpg)

*Theory board — sequential launches vs device arrays of pointers + $(M_g,N_g,K_g)$ in one kernel.*

### Pitfalls

- torch.bmm needs identical shapes — not a drop-in for MoE.
- “Grouped” in tutorial 03 ≠ Group GEMM in tutorial 08.
- Sequential launches hide true Group-GEMM benefits on many tiny problems.

### Self-check

1. List metadata needed for one-launch Group GEMM.
2. When do G host launches dominate runtime?
3. Sketch a pointer table for 3 experts with different M.


In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import importlib.util

import pytest
import torch

def _active_device():
    """Pick Triton active device, else CUDA, else CPU (study helper; no common/ package)."""
    try:
        import triton

        try:
            return triton.runtime.driver.active.get_active_torch_device()
        except Exception:
            pass
    except Exception:
        pass
    return torch.device("cuda" if torch.cuda.is_available() else "cpu")


DEVICE = _active_device()


def require_gpu() -> None:
    if not torch.cuda.is_available() and getattr(DEVICE, "type", None) != "cuda":
        pytest.skip(f"GPU not available (active device: {DEVICE})")


_matmul_path = Path(__file__).resolve().parents[1] / "03_matrix_multiplication" / "tutorial.py"
_spec = importlib.util.spec_from_file_location("matmul_mod", _matmul_path)
_matmul_mod = importlib.util.module_from_spec(_spec)
assert _spec.loader is not None
_spec.loader.exec_module(_matmul_mod)
matmul = _matmul_mod.matmul


## 1–2. Ladder: Naive sequential → Triton sequential → one-launch Group GEMM

For each $g$, independently,

$$
C^{(g)}_{mn} = \sum_k A^{(g)}_{mk} B^{(g)}_{kn}.
$$

| Stage | How work is launched | Algebra |
|---|---|---|
| **Naive** | Python `for` + `torch.matmul` | Exact $C^{(g)}=A^{(g)}B^{(g)}$ |
| **Intermediate** | Same loop + tutorial-03 `matmul` | Same $C^{(g)}$; still $G$ grids |
| **Triton Group GEMM** | One grid; device pointer / size tables | Same $C^{(g)}$; amortized launch |

**Proposition.** Running `torch.matmul` or tutorial-03 `matmul` in a Python `for` loop yields the Cartesian product of the individual solutions — there is no coupling between problems.

*Proof.* The maps $(A^{(g)},B^{(g)}) \mapsto C^{(g)}$ act on disjoint data; composition of independent functions is the tuple of results.

**Pointer table (official rung).** Host packs

$$
\texttt{A\_ptrs}[g],\;\texttt{B\_ptrs}[g],\;\texttt{C\_ptrs}[g],\;M[g],\;N[g],\;K[g]
$$

on device. Each program reads its group's metadata and runs the familiar $B_M\times B_N$ tile body (K-panels + `tl.dot`) from tutorial 03.

The remaining optimization is **scheduling** (one grid vs $G$ grids / persistent queues in 09), not algebra.


In [ ]:
def naive_group_gemm_pseudo(problems):
    """
    Sequential torch.matmul for each (A, B) pair.

    problems: list of (A, B) with A:[Mi, Ki], B:[Ki, Ni]

    Each call is a separate cuBLAS/torch launch. Fine for a few large GEMMs;
    painful when you have hundreds of tiny ones (launch overhead dominates).
    """
    return [torch.matmul(a, b) for a, b in problems]


In [ ]:
def group_gemm_sequential(problems):
    """
    Same loop, but each GEMM uses tutorial 03's Triton grouped matmul.
    (See markdown cells above for the math.)
    """
    return [matmul(a, b) for a, b in problems]


In [ ]:
def test_pseudo_matches_torch() -> None:
    torch.manual_seed(0)
    problems = [
        (torch.randn(32, 16), torch.randn(16, 24)),
        (torch.randn(8, 32), torch.randn(32, 8)),
    ]
    outs = naive_group_gemm_pseudo(problems)
    for (a, b), c in zip(problems, outs):
        assert torch.allclose(c, a @ b, atol=1e-5)


def test_correctness() -> None:
    require_gpu()
    problems = [
        (
            torch.randn(128, 64, device=DEVICE, dtype=torch.float16),
            torch.randn(64, 256, device=DEVICE, dtype=torch.float16),
        ),
        (
            torch.randn(64, 128, device=DEVICE, dtype=torch.float16),
            torch.randn(128, 32, device=DEVICE, dtype=torch.float16),
        ),
    ]
    outs = group_gemm_sequential(problems)
    for (a, b), c in zip(problems, outs):
        ref = torch.matmul(a, b)
        assert torch.allclose(c.float(), ref.float(), atol=1e-2, rtol=1e-2)
    print("group gemm (sequential) OK", len(outs))
